# Bronze → Silver

Parse kiểu → tra dimension (LEFT JOIN) → gán **một** `rule_id` cho dòng vi phạm → dòng sạch vào
`fact_valuation`, dòng lỗi vào `quarantine` → đối soát `fact + quarantine = Bronze active`.

In [1]:
import csv, json, logging, shutil, sys
from pathlib import Path

import pandas as pd

# Thư mục gốc dự án = nơi có src/config.py (kernel có thể mở với cwd bất kỳ)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
for p in (ROOT, ROOT / "tests"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import bronze, config, gold, reference, silver

logging.basicConfig(level=logging.INFO, format="%(levelname)-5s %(name)-7s %(message)s",
                    stream=sys.stdout, force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)
pd.set_option("display.max_columns", 40)

SANDBOX = ROOT / "notebook" / "_sandbox"             # mọi thứ notebook ghi ra nằm ở đây
SB_BRONZE = SANDBOX / "bronze" / "business_valuation"
SB_CONTROL = SANDBOX / "control" / "loaded_batches.csv"
SB_SILVER = SANDBOX / "silver"
REF_COPY = SANDBOX / "ref_model_map_bo_sung.csv"     # bản sao ref đã bổ sung (mục 3.8)

# Số đo trên 18 lô thật trong tài liệu — để SO, không assert (dữ liệu đổi thì số đổi)
EXPECTED = {
    "bronze": 52_791, "fact": 44_748, "quarantine": 8_043,
    "rules": {"CORRUPT_RECORD": 0, "DUP_KEY": 0, "MISSING_FIELD": 34, "DATE_INVALID": 225,
              "OUT_OF_SCOPE": 0, "MODEL_CODE_UNKNOWN": 5_154, "VALUE_INVALID": 280,
              "PRICE_INVALID": 2_350},
}

Hàm tiện ích dùng trong notebook này:

In [2]:
def fresh(*paths: Path) -> None:
    """Xoá các thư mục/file con của sandbox — notebook chỉ dọn phần của mình."""
    for p in paths:
        if SANDBOX not in p.parents:
            raise ValueError(f"{p} nằm ngoài sandbox — không xoá")
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists():
            p.unlink()
    SANDBOX.mkdir(parents=True, exist_ok=True)


def show(df, n: int = 10) -> pd.DataFrame:
    """DataFrame Spark -> pandas để notebook hiển thị thành bảng."""
    return df.limit(n).toPandas()


def compare(actual: dict, expected: dict) -> pd.DataFrame:
    """Bảng so số thực tế với số đo ghi trong tài liệu."""
    keys = list(expected) + [k for k in actual if k not in expected]
    return pd.DataFrame([{"mục": k, "thực tế": actual.get(k, 0), "tài liệu": expected.get(k, "—"),
                          "khớp": actual.get(k, 0) == expected.get(k)} for k in keys])


def ensure_bronze(spark) -> None:
    """Sandbox chưa có Bronze (chưa chạy notebook 02) -> nạp 18 lô."""
    if not SB_BRONZE.exists():
        print("Sandbox chưa có Bronze → nạp 18 lô (như notebook 02)")
        bronze.run(spark, config.LANDING_DIR, SB_BRONZE, SB_CONTROL)


fresh(SB_SILVER, SANDBOX / "silver_bo_sung", REF_COPY)

In [3]:
from pyspark.sql import functions as F
from src.spark import get_spark

spark = get_spark("03_silver")
print("Spark", spark.version)

ensure_bronze(spark)

Spark 3.5.9


## mỗi tháng lấy bản nạp mới nhất. 
Nếu trong bronze đã chạy bước gửi lại lô, tháng `2026-06` có **2 bản nạp** trong Bronze nhưng chỉ **1** bản active.

In [4]:
# đọc bronze và chọn bản active
bz = spark.read.parquet(SB_BRONZE.as_posix()) 
active = silver.active_bronze(bz).cache()
print(f"Bronze: {bz.count()} dòng · active: {active.count()} dòng")

# thống kê theo tháng, số bản nạp (2026-06 sẽ có 2 bản)
per_month = (bz.groupBy("as_of_month").agg(F.countDistinct("_batch_id").alias("so_ban_nap"),
                                           F.count("*").alias("dong_bronze"))
               .join(active.groupBy("as_of_month").agg(F.first("_batch_id").alias("ban_active"),
                                                       F.countDistinct("_batch_id").alias("n_active"),
                                                       F.count("*").alias("dong_active")), "as_of_month")
               .orderBy("as_of_month").toPandas())

assert (per_month["n_active"] == 1).all(), "Mỗi tháng chỉ được có 1 bản active" #active=1
per_month.drop(columns="n_active")

Bronze: 62815 dòng · active: 59421 dòng


,as_of_month,so_ban_nap,dong_bronze,ban_active,dong_active
0,2025-01,1,3119,bvv_backfill_202501_01,3119
1,2025-02,1,1982,bvv_backfill_202502_01,1982
2,2025-03,1,2650,bvv_backfill_202503_01,2650
3,2025-04,1,2417,bvv_backfill_202504_01,2417
4,2025-05,1,2422,bvv_backfill_202505_01,2422
5,2025-06,1,2507,bvv_backfill_202506_01,2507
6,2025-07,1,2775,bvv_backfill_202507_01,2775
7,2025-08,1,2333,bvv_backfill_202508_01,2333
8,2025-09,1,2279,bvv_backfill_202509_01,2279
9,2025-10,1,3418,bvv_backfill_202510_01,3418


## Dựng dimension từ `ref_*`

In [5]:
# tạo dim model, kiểm tra ma_dong_xe không trùng và tạo model_key để nối với fact. 
# Grain: hang_xe, hieu_xe, ma_kieu, ma_dong_xe
dim_model = silver.build_dim_model(silver.read_ref_csv(spark, reference.resolve(config.REF_MODEL_MAP_NAME))).cache()
dim_model

DataFrame[model_key: bigint, ma_dong_xe: string, ma_hieu: string, hang_xe: string, hieu_xe: string]

In [6]:
# tạo dim_vehicle_type. Grain = nhom_xe
dim_vehicle_type = silver.build_dim_vehicle_type(silver.read_ref_csv(spark, reference.resolve(config.REF_VEH_TYPE_NAME))).cache()
print("dòng · dim_vehicle_type:", dim_vehicle_type.count())

dòng · dim_vehicle_type: 42


In [7]:
check_dim_model = silver.build_dim_model(silver.read_ref_csv(spark, reference.resolve(config.REF_MODEL_MAP_NAME)))
# kiểm tra khóa băm cho cùng 1 mã thì luôn ra cùng 1 số
assert dim_model.select("ma_dong_xe", "model_key").subtract(check_dim_model.select("ma_dong_xe", "model_key")).count() == 0

In [8]:
display(show(dim_model, 5))
show(dim_vehicle_type, 30)

,model_key,ma_dong_xe,ma_hieu,hang_xe,hieu_xe
0,-6120385611582633742,ACU01001,HYU01,Hyundai,SantaFe
1,5098237438784397224,AUD09001,AUD09,Audi,Q5
2,4800818618693316909,AUD09002,AUD09,Audi,Q5
3,8087312717288551251,AUD09003,AUD09,Audi,Q5
4,-7326266216587769992,AUD09004,AUD09,Audi,Q5


,vehicle_type_key,nhom_xe,mo_ta,muc_dich_su_dung,trong_pham_vi
0,5973645975235149644,KKD1,Non Commercial Motor Vehicle(0-16),Không kinh doanh,True
1,-3026751655632761104,KKD1-1,Non Commercial Motor Vehicle (0-5),Không kinh doanh,True
2,-646254655641179225,KKD1-2,Non Commercial Motor Vehicle (6-8),Không kinh doanh,True
3,8268081440573965364,KKD3,Non Commercial Motor Vehicle (>16),Không kinh doanh,True
4,-6496754862050172983,TAXI,Taxi truyền thống,Taxi,True
5,5913661108233320170,TAXI-1,Xe taxi truyền thống (≤ 5 chỗ),Taxi,True
6,2865103113153016831,TAXI-2,Xe taxi công nghệ,Taxi,True
7,-9138903737752284776,XEKD1,Commercial car (others),Kinh doanh vận tải,True
8,-1276363807411862068,XEKD2,Commercial Motor Vehicle,Kinh doanh vận tải,True
9,4189373175316371494,XTTL,Self-drive rental car (incl. Grab),Cho thuê tự lái,True


In [9]:
from tests.helpers import VALID, bronze_df

ma_xe = dim_model.first()["ma_dong_xe"]                               # mã xe có thật trong dim
nhom = dim_vehicle_type.filter("trong_pham_vi").first()["nhom_xe"]    # loại xe trong phạm vi
base = {**VALID, "ma_dong_xe": ma_xe, "nhom_xe": nhom}

# test case
CASE_LIST = [
    ("hợp lệ",                               "BV-OK",     {},                                      None),
    ("vỡ cột",                               "BV-CORRUPT", {},                                     "CORRUPT_RECORD"),
    ("khoá rỗng",                            "",          {},                                      "DUP_KEY"),
    ("trùng khoá BV-OK, hiệu lực sớm hơn",   "BV-OK",     {"hieu_luc_tu": "2026-06-11"},           "DUP_KEY"),
    ("thiếu số chỗ (toàn khoảng trắng)",     "BV-MISS",   {"so_cho_ngoi": "  "},                   "MISSING_FIELD"),
    ("ngày duyệt sai dạng",                  "BV-DATE1",  {"ngay_duyet": "10/06/2026"},            "DATE_INVALID"),
    ("hiệu lực đến < từ",                    "BV-DATE2",  {"hieu_luc_den": "2026-01-01"},          "DATE_INVALID"),
    ("hiệu lực lệch ngày duyệt > 366 ngày",  "BV-DATE3",  {"hieu_luc_tu": "2024-06-12",
                                                           "hieu_luc_den": "2025-06-12"},          "DATE_INVALID"),
    ("loại xe không có trong dim",           "BV-SCOPE",  {"nhom_xe": "KHONG_CO"},                 "OUT_OF_SCOPE"),
    ("mã xe không có trong dim",             "BV-MODEL",  {"ma_dong_xe": "ZZZ99999"},              "MODEL_CODE_UNKNOWN"),
    ("đời xe > năm duyệt + 1",               "BV-VAL1",   {"nam_san_xuat": "2030"},                "VALUE_INVALID"),
    ("năm sản xuất không phải số",           "BV-VAL2",   {"nam_san_xuat": "abc"},                 "VALUE_INVALID"),
    ("loai_dinh_gia lạ",                     "BV-VAL3",   {"loai_dinh_gia": "demo"},               "VALUE_INVALID"),
    ("giá 0",                                "BV-PRICE1", {"gia_tri_dinh_gia": "0"},               "PRICE_INVALID"),
    ("giá viết kiểu 1.669.000.000",          "BV-PRICE2", {"gia_tri_dinh_gia": "1.669.000.000"},   "PRICE_INVALID"),
    ("mã xe lạ + giá 0 → luật đứng trước",   "BV-PRIO",   {"ma_dong_xe": "ZZZ99999",
                                                           "gia_tri_dinh_gia": "0"},               "MODEL_CODE_UNKNOWN"),
]

In [10]:
# loại xe có trong dim nhưng ngoài phạm vi
out_of_scope = dim_vehicle_type.filter(~F.col("trong_pham_vi")).first()
if out_of_scope:                                                     
    CASE_LIST.append(("loại xe trong dim nhưng ngoài phạm vi", "BV-SCOPE2",
                      {"nhom_xe": out_of_scope["nhom_xe"]}, "OUT_OF_SCOPE"))
print("Loại xe ngoài phạm vi dùng để thử:", out_of_scope)
print("Số tình huống:", len(CASE_LIST))

Loại xe ngoài phạm vi dùng để thử: Row(vehicle_type_key=-4064358858473899440, nhom_xe='BUYT', mo_ta='Bus', muc_dich_su_dung='Khác', trong_pham_vi=False)
Số tình huống: 17


In [11]:
rows_t = [{**base, **changes, "ma_dinh_gia": key, "_batch_id": case} for case, key, changes, _ in CASE_LIST]
pd.DataFrame([{"tình huống": r["_batch_id"],
               "ô bị đổi": {k: v for k, v in r.items() if base.get(k) != v and k != "_batch_id"}}
              for r in rows_t])

,tình huống,ô bị đổi
0,hợp lệ,{'ma_dinh_gia': 'BV-OK'}
1,vỡ cột,{'ma_dinh_gia': 'BV-CORRUPT'}
2,khoá rỗng,{'ma_dinh_gia': ''}
3,"trùng khoá BV-OK, hiệu lực sớm hơn","{'hieu_luc_tu': '2026-06-11', 'ma_dinh_gia': '..."
4,thiếu số chỗ (toàn khoảng trắng),"{'so_cho_ngoi': ' ', 'ma_dinh_gia': 'BV-MISS'}"
5,ngày duyệt sai dạng,"{'ngay_duyet': '10/06/2026', 'ma_dinh_gia': 'B..."
6,hiệu lực đến < từ,"{'hieu_luc_den': '2026-01-01', 'ma_dinh_gia': ..."
7,hiệu lực lệch ngày duyệt > 366 ngày,"{'hieu_luc_tu': '2024-06-12', 'hieu_luc_den': ..."
8,loại xe không có trong dim,"{'nhom_xe': 'KHONG_CO', 'ma_dinh_gia': 'BV-SCO..."
9,mã xe không có trong dim,"{'ma_dong_xe': 'ZZZ99999', 'ma_dinh_gia': 'BV-..."


In [12]:
df_cases = bronze_df(spark, rows_t).withColumn(
    bronze.CORRUPT_COL, F.when(F.col("_batch_id") == "vỡ cột", F.lit("dòng,thô,bị,vỡ")))
show(df_cases.select("_batch_id", "ma_dinh_gia", "ma_dong_xe", "nhom_xe",
                     "gia_tri_dinh_gia", bronze.CORRUPT_COL), 30)

,_batch_id,ma_dinh_gia,ma_dong_xe,nhom_xe,gia_tri_dinh_gia,_corrupt_record
0,hợp lệ,BV-OK,ACU01001,KKD1,500000000,None
1,vỡ cột,BV-CORRUPT,ACU01001,KKD1,500000000,"dòng,thô,bị,vỡ"
2,khoá rỗng,,ACU01001,KKD1,500000000,None
3,"trùng khoá BV-OK, hiệu lực sớm hơn",BV-OK,ACU01001,KKD1,500000000,None
4,thiếu số chỗ (toàn khoảng trắng),BV-MISS,ACU01001,KKD1,500000000,None
5,ngày duyệt sai dạng,BV-DATE1,ACU01001,KKD1,500000000,None
6,hiệu lực đến < từ,BV-DATE2,ACU01001,KKD1,500000000,None
7,hiệu lực lệch ngày duyệt > 366 ngày,BV-DATE3,ACU01001,KKD1,500000000,None
8,loại xe không có trong dim,BV-SCOPE,ACU01001,KHONG_CO,500000000,None
9,mã xe không có trong dim,BV-MODEL,ZZZ99999,KKD1,500000000,None


In [13]:
# transform trên Silver
fact_t, quar_t = silver.transform(df_cases, dim_model, dim_vehicle_type)
print(f"fact: {fact_t.count()} · quarantine: {quar_t.count()} · đầu vào: {df_cases.count()}")
display(show(quar_t.select("_batch_id", "ma_dinh_gia", "rule_id"), 30))
show(fact_t, 5)


fact: 1 · quarantine: 16 · đầu vào: 17


,_batch_id,ma_dinh_gia,rule_id
0,khoá rỗng,,DUP_KEY
1,vỡ cột,BV-CORRUPT,CORRUPT_RECORD
2,ngày duyệt sai dạng,BV-DATE1,DATE_INVALID
3,hiệu lực đến < từ,BV-DATE2,DATE_INVALID
4,hiệu lực lệch ngày duyệt > 366 ngày,BV-DATE3,DATE_INVALID
5,thiếu số chỗ (toàn khoảng trắng),BV-MISS,MISSING_FIELD
6,mã xe không có trong dim,BV-MODEL,MODEL_CODE_UNKNOWN
7,"trùng khoá BV-OK, hiệu lực sớm hơn",BV-OK,DUP_KEY
8,giá 0,BV-PRICE1,PRICE_INVALID
9,giá viết kiểu 1.669.000.000,BV-PRICE2,PRICE_INVALID


,ma_dinh_gia,model_key,vehicle_type_key,ngay_duyet_key,hieu_luc_tu_key,nam_san_xuat,so_cho_ngoi,loai_dinh_gia,tuoi_xe,gia_tri_trieu_vnd,ngay_duyet,hieu_luc_tu,hieu_luc_den,as_of_month,_batch_id
0,BV-OK,-6120385611582633742,5973645975235149644,20260610,20260612,2020,5,used,6,500.0,2026-06-10,2026-06-12,2027-06-12,2026-06,hợp lệ


In [14]:
in_quar_t = {r["_batch_id"]: r["rule_id"] for r in quar_t.select("_batch_id", "rule_id").collect()}
in_fact = {r["_batch_id"] for r in fact_t.select("_batch_id").collect()}
print("Quarantine:", in_quar_t)
print("*****************************************")
print("Vào fact:", in_fact)

Quarantine: {'khoá rỗng': 'DUP_KEY', 'vỡ cột': 'CORRUPT_RECORD', 'ngày duyệt sai dạng': 'DATE_INVALID', 'hiệu lực đến < từ': 'DATE_INVALID', 'hiệu lực lệch ngày duyệt > 366 ngày': 'DATE_INVALID', 'thiếu số chỗ (toàn khoảng trắng)': 'MISSING_FIELD', 'mã xe không có trong dim': 'MODEL_CODE_UNKNOWN', 'trùng khoá BV-OK, hiệu lực sớm hơn': 'DUP_KEY', 'giá 0': 'PRICE_INVALID', 'giá viết kiểu 1.669.000.000': 'PRICE_INVALID', 'mã xe lạ + giá 0 → luật đứng trước': 'MODEL_CODE_UNKNOWN', 'loại xe không có trong dim': 'OUT_OF_SCOPE', 'loại xe trong dim nhưng ngoài phạm vi': 'OUT_OF_SCOPE', 'đời xe > năm duyệt + 1': 'VALUE_INVALID', 'năm sản xuất không phải số': 'VALUE_INVALID', 'loai_dinh_gia lạ': 'VALUE_INVALID'}
*****************************************
Vào fact: {'hợp lệ'}


In [15]:
table = pd.DataFrame([{"tình huống": case, "ma_dinh_gia": key, "mong đợi": exp or "→ fact",
                       "thực tế": in_quar_t.get(case) or ("→ fact" if case in in_fact else "MẤT DÒNG")}
                      for case, key, _, exp in CASE_LIST])
table["khớp"] = table["mong đợi"] == table["thực tế"]
table[~table["khớp"]]

,tình huống,ma_dinh_gia,mong đợi,thực tế,khớp


In [16]:
assert table["khớp"].all(), "Có dòng đi sai chỗ"
assert fact_t.count() + quar_t.count() == df_cases.count(), "Có dòng biến mất"
print(f"{fact_t.count()} + {quar_t.count()} = {fact_t.count() + quar_t.count()}  vs  {df_cases.count()}")

1 + 16 = 17  vs  17


## Chạy luật trên dữ liệu thật (chưa ghi)

In [17]:
# parse kiểu, tra dimension và assign rule
staged = silver.assign_rule_id(silver.parse_and_lookup(active, dim_model, dim_vehicle_type)).cache()
# Cột p_ đã parse cạnh cột thô, kèm khoá dim và rule_id
show(staged.select("ma_dinh_gia", "gia_tri_dinh_gia", "p_gia_vnd", "ngay_duyet", "p_ngay_duyet",
                   "ma_dong_xe", "model_key", "nhom_xe", "vehicle_type_key", "rule_id"), 10)
# Parse thất bại, record có giá trị nhưng p_ là null
show(staged.filter(F.col("p_gia_vnd").isNull() & F.col("gia_tri_dinh_gia").isNotNull())
           .select("gia_tri_dinh_gia", "p_gia_vnd", "rule_id"), 10)

,gia_tri_dinh_gia,p_gia_vnd,rule_id


In [18]:
by_rule = {r["rule_id"]: r["count"] for r in staged.groupBy("rule_id").count().collect()}
by_rule

{'MODEL_CODE_UNKNOWN': 3291,
 None: 53064,
 'PRICE_INVALID': 2513,
 'MISSING_FIELD': 34,
 'DATE_INVALID': 231,
 'VALUE_INVALID': 288}

In [19]:
# dòng đã làm sạch
n_clean = by_rule.pop(None, 0)
print(f"fact {n_clean} · quarantine {sum(by_rule.values())} (tài liệu: ...)")
print(n_clean + sum(by_rule.values()), "==", active.count())

fact 53064 · quarantine 6357 (tài liệu: ...)
59421 == 59421


In [20]:
# so sánh trên từng luật
compare(by_rule, EXPECTED["rules"])
for rule in silver.RULES:
    print(f"── {rule}: {silver.RULES[rule]}")
    display(show(staged.filter(F.col("rule_id") == rule)
                       .select("ma_dinh_gia", "ma_dong_xe", "nhom_xe", "nam_san_xuat",
                               "gia_tri_dinh_gia", "ngay_duyet", "hieu_luc_tu", "hieu_luc_den"), 3))

── CORRUPT_RECORD: record vỡ cột, không parse được


,ma_dinh_gia,ma_dong_xe,nhom_xe,nam_san_xuat,gia_tri_dinh_gia,ngay_duyet,hieu_luc_tu,hieu_luc_den


── DUP_KEY: ma_dinh_gia rỗng hoặc trùng lặp (giữ 1 record, các record còn lại vào đây)


,ma_dinh_gia,ma_dong_xe,nhom_xe,nam_san_xuat,gia_tri_dinh_gia,ngay_duyet,hieu_luc_tu,hieu_luc_den


── MISSING_FIELD: rỗng tại ma_dong_xe / nam_san_xuat / so_cho_ngoi


...

,ma_dinh_gia,ma_dong_xe,nhom_xe,nam_san_xuat,gia_tri_dinh_gia,ngay_duyet,hieu_luc_tu,hieu_luc_den
0,BV-003474c3f94c3a07,THA238004,XEKD1,2021,940000000,2026-08-24,2026-08-24,2027-08-24
1,BV-00e067ecdd96cc08,NIS08001,KKD1-1,2018,250000000,2025-08-20,2025-07-31,2026-07-31
2,BV-010fd1b0679604e8,VOL05004,KKD1-1,2022,1600000000,2025-01-23,2025-01-27,2026-01-27


── VALUE_INVALID: nam_san_xuat / so_cho_ngoi / loai_dinh_gia ngoài miền giá trị (domain) hợp lệ


,ma_dinh_gia,ma_dong_xe,nhom_xe,nam_san_xuat,gia_tri_dinh_gia,ngay_duyet,hieu_luc_tu,hieu_luc_den
0,BV-0b6dd633615eab94,VIN028004,KKD1-1,2025,0,2025-12-30,2025-12-30,2025-12-30
1,BV-126db81f10d402c5,VIN028002,KKD1-1,2026,0,2026-04-28,2026-04-27,2026-04-27
2,BV-28a9194dcec1318a,VIN034001,XEKD2,2026,0,2026-03-20,2026-03-20,2026-03-20


── PRICE_INVALID: gia_tri_dinh_gia sai định dạng hoặc ngoài [50 triệu, 50 tỷ] VND


,ma_dinh_gia,ma_dong_xe,nhom_xe,nam_san_xuat,gia_tri_dinh_gia,ngay_duyet,hieu_luc_tu,hieu_luc_den
0,BV-00df9fc81c88ffe0,HON03002,KKD1-1,2019,0,2025-02-07,2025-01-07,2025-01-07
1,BV-01769e344794dd25,HYU01007,KKD1-2,2021,0,2026-01-20,2026-01-21,2026-01-21
2,BV-01dad6744d073a3b,MAZ06010,KKD1-1,2022,0,2025-05-29,2025-05-22,2025-05-22


In [21]:
parsed = silver.parse_and_lookup(active, dim_model, dim_vehicle_type)
flags = parsed.select([c.alias(r) for r, c in silver.rule_conditions(parsed).items()])

In [22]:
# Phân bố: bao nhiêu dòng vi phạm 0, 1, 2, 3... luật
n_viol = sum(F.col(r).cast("int") for r in silver.RULES)
show(flags.groupBy(n_viol.alias("so_luat_vi_pham")).count().orderBy("so_luat_vi_pham"))

,so_luat_vi_pham,count
0,0,53064
1,1,5801
2,2,501
3,3,38
4,4,17


In [23]:
viol = flags.select([F.count_if(F.col(r)).alias(r) for r in silver.RULES]).first().asDict()
viol

{'CORRUPT_RECORD': 0,
 'DUP_KEY': 0,
 'MISSING_FIELD': 34,
 'DATE_INVALID': 253,
 'OUT_OF_SCOPE': 0,
 'MODEL_CODE_UNKNOWN': 3341,
 'VALUE_INVALID': 340,
 'PRICE_INVALID': 3017}

In [24]:
pd.DataFrame({"vi phạm (đếm độc lập)": viol,
              "mang rule_id (sau ưu tiên)": {r: by_rule.get(r, 0) for r in silver.RULES}})
assert all(viol[r] >= by_rule.get(r, 0) for r in silver.RULES), "sau ưu tiên không thể lớn hơn đếm độc lập"
first_rule = next(iter(silver.RULES))
assert viol[first_rule] == by_rule.get(first_rule, 0), "luật đầu tiên không bị luật nào lấy dòng"

n_any = flags.filter(" OR ".join(silver.RULES)).count()   # số dòng vi phạm ít nhất 1 luật
print(n_any, "==", sum(by_rule.values()), "(= quarantine)")

6357 == 6357 (= quarantine)


In [25]:
fl = staged.select("rule_id", *[c.alias(r) for r, c in silver.rule_conditions(staged).items()])
show(fl.filter("PRICE_INVALID").groupBy("rule_id").count().orderBy(F.desc("count")))

,rule_id,count
0,PRICE_INVALID,2513
1,VALUE_INVALID,267
2,MODEL_CODE_UNKNOWN,204
3,DATE_INVALID,32
4,MISSING_FIELD,1


In [26]:
res_silver = silver.run(spark, SB_BRONZE, SB_SILVER)
compare(res_silver, {"bronze": EXPECTED["bronze"], "fact": EXPECTED["fact"],
                     "quarantine": EXPECTED["quarantine"]})

INFO  silver  Ref: ref_model_map        version=2026-07 · 2167 record · sha256 777979e4299e…
INFO  silver  Ref: ref_veh_type_scope   version=2026-06 · 42 record · sha256 709f4101b488…
INFO  silver  Silver: bronze active 59421 = fact 53064 + quarantine 6357 (10.70%)
INFO  silver          MODEL_CODE_UNKNOWN     3291
INFO  silver          PRICE_INVALID          2513
INFO  silver          VALUE_INVALID           288
INFO  silver          DATE_INVALID            231
INFO  silver          MISSING_FIELD            34
INFO  silver          dim_model 2167 · dim_vehicle_type 42 · dim_date 1461


,mục,thực tế,tài liệu,khớp
0,bronze,59421,52791,False
1,fact,53064,44748,False
2,quarantine,6357,8043,False
3,ref_versions,"{'ref_model_map': '2026-07', 'ref_veh_type_sco...",—,False


In [27]:
# run layer silve
res_silver = silver.run(spark, SB_BRONZE, SB_SILVER)
for p in sorted(SB_SILVER.rglob("*.parquet")):
    print(p.relative_to(SB_SILVER), f"{p.stat().st_size/1024:.0f} KB")

INFO  silver  Ref: ref_model_map        version=2026-07 · 2167 record · sha256 777979e4299e…
INFO  silver  Ref: ref_veh_type_scope   version=2026-06 · 42 record · sha256 709f4101b488…
INFO  silver  Silver: bronze active 59421 = fact 53064 + quarantine 6357 (10.70%)
INFO  silver          MODEL_CODE_UNKNOWN     3291
INFO  silver          PRICE_INVALID          2513
INFO  silver          VALUE_INVALID           288
INFO  silver          DATE_INVALID            231
INFO  silver          MISSING_FIELD            34
INFO  silver          dim_model 2167 · dim_vehicle_type 42 · dim_date 1461
dim_date\part-00000-0937cccd-89d3-4d5c-aba8-869900b173ce-c000.snappy.parquet 14 KB
dim_model\part-00000-638cd3da-63c0-43a3-90ee-58a15fdbad09-c000.snappy.parquet 30 KB
dim_vehicle_type\part-00000-eef5e9d6-8126-4959-942a-a970a451ca27-c000.snappy.parquet 3 KB
fact_valuation\as_of_month=2025-01\part-00001-8808d40a-0fb9-4317-9f48-42ec58168ce0.c000.snappy.parquet 85 KB
fact_valuation\as_of_month=2025-02\part-000

In [28]:
compare(res_silver, {"bronze": EXPECTED["bronze"], "fact": EXPECTED["fact"],
                     "quarantine": EXPECTED["quarantine"]})
assert res_silver["fact"] == n_clean and res_silver["quarantine"] == sum(by_rule.values())
assert res_silver["bronze"] == active.count()
for name in ["fact_valuation", "quarantine", "dim_model", "dim_vehicle_type", "dim_date"]:
    print(f"{name:18s}", spark.read.parquet((SB_SILVER / name).as_posix()).count())

fact_valuation     53064
quarantine         6357
dim_model          2167
dim_vehicle_type   42
dim_date           1461


In [29]:
# kiểm tra số dòng fact & quarantine theo từng tháng, đọc từ cột partition
fq = (spark.read.parquet((SB_SILVER / "fact_valuation").as_posix()).groupBy("as_of_month").count()
        .withColumnRenamed("count", "fact")
        .join(spark.read.parquet((SB_SILVER / "quarantine").as_posix()).groupBy("as_of_month").count()
                .withColumnRenamed("count", "quarantine"), "as_of_month", "full")
        .orderBy("as_of_month"))
show(fq, 30)

,as_of_month,fact,quarantine
0,2025-01,2819,300
1,2025-02,1714,268
2,2025-03,2318,332
3,2025-04,2114,303
4,2025-05,2117,305
5,2025-06,2197,310
6,2025-07,2415,360
7,2025-08,2061,272
8,2025-09,2020,259
9,2025-10,3106,312


In [30]:
assert silver.run(spark, SB_BRONZE, SB_SILVER) == res_silver

INFO  silver  Ref: ref_model_map        version=2026-07 · 2167 record · sha256 777979e4299e…
INFO  silver  Ref: ref_veh_type_scope   version=2026-06 · 42 record · sha256 709f4101b488…
INFO  silver  Silver: bronze active 59421 = fact 53064 + quarantine 6357 (10.70%)
INFO  silver          MODEL_CODE_UNKNOWN     3291
INFO  silver          PRICE_INVALID          2513
INFO  silver          VALUE_INVALID           288
INFO  silver          DATE_INVALID            231
INFO  silver          MISSING_FIELD            34
INFO  silver          dim_model 2167 · dim_vehicle_type 42 · dim_date 1461


## Kiểm tính bất biến trên các bảng đã ghi

In [31]:
T = {name: spark.read.parquet((SB_SILVER / name).as_posix()) for name in gold.SILVER_TABLES}
fact, quar, dim_date = T["fact_valuation"], T["quarantine"], T["dim_date"]


def orphans(key: str, dim, dim_key: str) -> int:
    """Số dòng fact có khoá ngoại không tìm thấy trong dimension."""
    return fact.join(dim.withColumnRenamed(dim_key, key), key, "left_anti").count()


checks = pd.DataFrame([
    ("fact + quarantine = Bronze active", fact.count() + quar.count() == active.count()),
    ("ma_dinh_gia duy nhất trong fact", fact.select("ma_dinh_gia").distinct().count() == fact.count()),
    ("mọi model_key có trong dim_model", orphans("model_key", T["dim_model"], "model_key") == 0),
    ("mọi vehicle_type_key có trong dim_vehicle_type",
     orphans("vehicle_type_key", T["dim_vehicle_type"], "vehicle_type_key") == 0),
    ("mọi ngay_duyet_key có trong dim_date", orphans("ngay_duyet_key", dim_date, "date_key") == 0),
    ("mọi hieu_luc_tu_key có trong dim_date", orphans("hieu_luc_tu_key", dim_date, "date_key") == 0),
    ("quarantine = cột thô Bronze + rule_id", set(quar.columns) == set(active.columns) | {"rule_id"}),
    ("mọi dòng quarantine có rule_id", quar.filter(F.col("rule_id").isNull()).count() == 0),
], columns=["bất biến", "đạt"])

assert checks["đạt"].all()
fact.printSchema()
checks

root
 |-- ma_dinh_gia: string (nullable = true)
 |-- model_key: long (nullable = true)
 |-- vehicle_type_key: long (nullable = true)
 |-- ngay_duyet_key: integer (nullable = true)
 |-- hieu_luc_tu_key: integer (nullable = true)
 |-- nam_san_xuat: integer (nullable = true)
 |-- so_cho_ngoi: integer (nullable = true)
 |-- loai_dinh_gia: string (nullable = true)
 |-- tuoi_xe: integer (nullable = true)
 |-- gia_tri_trieu_vnd: double (nullable = true)
 |-- ngay_duyet: date (nullable = true)
 |-- hieu_luc_tu: date (nullable = true)
 |-- hieu_luc_den: date (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- as_of_month: string (nullable = true)



,bất biến,đạt
0,fact + quarantine = Bronze active,True
1,ma_dinh_gia duy nhất trong fact,True
2,mọi model_key có trong dim_model,True
3,mọi vehicle_type_key có trong dim_vehicle_type,True
4,mọi ngay_duyet_key có trong dim_date,True
5,mọi hieu_luc_tu_key có trong dim_date,True
6,quarantine = cột thô Bronze + rule_id,True
7,mọi dòng quarantine có rule_id,True


In [32]:
display(show(fact, 5))
show(quar.groupBy("rule_id").count().orderBy(F.desc("count")))

,ma_dinh_gia,model_key,vehicle_type_key,ngay_duyet_key,hieu_luc_tu_key,nam_san_xuat,so_cho_ngoi,loai_dinh_gia,tuoi_xe,gia_tri_trieu_vnd,ngay_duyet,hieu_luc_tu,hieu_luc_den,_batch_id,as_of_month
0,BV-001b0f3307609b8a,3577285018243249725,-3026751655632761104,20251201,20260117,2021,5,used,4,679.0,2025-12-01,2026-01-17,2027-01-17,bvv_backfill_202512_01,2025-12
1,BV-00d1bd5e971b2c2a,1150268487476714469,-3026751655632761104,20251206,20251205,2024,5,used,1,550.0,2025-12-06,2025-12-05,2026-12-05,bvv_backfill_202512_01,2025-12
2,BV-00edd52989f8c0e8,-8209549347550812105,-3026751655632761104,20251208,20251203,2024,5,used,1,450.0,2025-12-08,2025-12-03,2026-12-03,bvv_backfill_202512_01,2025-12
3,BV-023193adb70bb41c,7197684704614915188,-3026751655632761104,20251208,20251209,2024,5,used,1,405.0,2025-12-08,2025-12-09,2026-12-09,bvv_backfill_202512_01,2025-12
4,BV-02351309bc62f1ee,-6214172385842306133,-3026751655632761104,20251227,20251227,2024,5,used,1,740.0,2025-12-27,2025-12-27,2026-12-27,bvv_backfill_202512_01,2025-12


,rule_id,count
0,MODEL_CODE_UNKNOWN,3291
1,PRICE_INVALID,2513
2,VALUE_INVALID,288
3,DATE_INVALID,231
4,MISSING_FIELD,34


## bổ sung `ref_model_map` rồi chạy lại Silver

Lấy mã xe chưa có trong bảng ref xuất hiện nhiều nhất, thêm vào **bản sao** của `ref_model_map.csv`
(hãng/dòng xe là giá trị thử), chạy lại Silver sang thư mục khác. Không sửa code, không cần bên cung
cấp gửi lại lô.

In [33]:
# kiểm tra ma dong xe bị thiếu nhiều nhất
top = (quar.filter("rule_id = 'MODEL_CODE_UNKNOWN'").groupBy("ma_dong_xe").count()
          .orderBy(F.desc("count")).first())
show(quar.filter("rule_id = 'MODEL_CODE_UNKNOWN'").groupBy("ma_dong_xe").count()
         .orderBy(F.desc("count")), 10)

,ma_dong_xe,count
0,VIN031001,100
1,VIN016001,95
2,LV001001,52
3,TOY20002,52
4,TOY38003,46
5,SUB07001,46
6,FOR17001,45
7,NIS30001,44
8,TRACO06001,41
9,MG03006,38


In [34]:
with open(reference.resolve(config.REF_MODEL_MAP_NAME), newline="", encoding="utf-8-sig") as f:
    ref_rows = list(csv.DictReader(f))
ref_rows.append({"ma_dong_xe": top["ma_dong_xe"].strip(), "ma_hieu": "THU01",
                 "hang_xe": "(hãng thử)", "hieu_xe": "(dòng thử)"})
with open(REF_COPY, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(ref_rows[0]))
    w.writeheader()
    w.writerows(ref_rows)

print(len(ref_rows) - 1, "→", len(ref_rows))
pd.read_csv(REF_COPY, encoding="utf-8-sig", dtype=str).tail(3)

2167 → 2168


,ma_dong_xe,ma_hieu,hang_xe,hieu_xe
2165,VOW20002,VOW20,Volkswagen,Viloran
2166,VOW20003,VOW20,Volkswagen,Viloran
2167,VIN031001,THU01,(hãng thử),(dòng thử)


In [35]:
res_rescue = silver.run(spark, SB_BRONZE, SANDBOX / "silver_bo_sung", ref_model_map=REF_COPY)
assert res_rescue["bronze"] == res_silver["bronze"]
assert res_rescue["fact"] + res_rescue["quarantine"] == res_rescue["bronze"]
print(f"quarantine {res_silver['quarantine']} → {res_rescue['quarantine']}")

INFO  silver  Ref: ref_model_map_bo_sung version=unversioned · 2168 record · sha256 cb7910ebc93d…
INFO  silver  Ref: ref_veh_type_scope   version=2026-06 · 42 record · sha256 709f4101b488…
INFO  silver  Silver: bronze active 59421 = fact 53154 + quarantine 6267 (10.55%)
INFO  silver          MODEL_CODE_UNKNOWN     3191
INFO  silver          PRICE_INVALID          2517
INFO  silver          VALUE_INVALID           294
INFO  silver          DATE_INVALID            231
INFO  silver          MISSING_FIELD            34
INFO  silver          dim_model 2168 · dim_vehicle_type 42 · dim_date 1461
quarantine 6357 → 6267


In [36]:
q2 = spark.read.parquet((SANDBOX / "silver_bo_sung" / "quarantine").as_posix())
print(f"fact {res_silver['fact']} → {res_rescue['fact']}  (+...)")

fact 53064 → 53154  (+...)


In [37]:
same_code = F.col("ma_dong_xe") == top["ma_dong_xe"]
pd.DataFrame({"trước": {r["rule_id"]: r["count"] for r in quar.filter(same_code).groupBy("rule_id").count().collect()},
              "sau bổ sung": {r["rule_id"]: r["count"] for r in q2.filter(same_code).groupBy("rule_id").count().collect()}}).fillna(0).astype(int)

,trước,sau bổ sung
MODEL_CODE_UNKNOWN,100,0
PRICE_INVALID,0,4
VALUE_INVALID,0,6


In [38]:
code = top["ma_dong_xe"]
before, after = quar.filter(same_code).count(), q2.filter(same_code).count()
print(f"rời quarantine: {before - after} · fact tăng: {res_rescue['fact'] - res_silver['fact']}")
assert before - after == res_rescue["fact"] - res_silver["fact"]

rời quarantine: 90 · fact tăng: 90


In [39]:
SB2 = SANDBOX / "silver_bo_sung"
fact2 = spark.read.parquet((SB2 / "fact_valuation").as_posix())
dim2 = spark.read.parquet((SB2 / "dim_model").as_posix())
print("dòng fact của mã mới:", fact2.join(dim2.filter(F.col("ma_dong_xe") == code.strip()), "model_key").count())

dòng fact của mã mới: 90


In [40]:
old = T["dim_model"].select("ma_dong_xe", "model_key")
assert old.subtract(dim2.select("ma_dong_xe", "model_key")).count() == 0   # mọi khoá cũ còn nguyên
print("dim_model:", old.count(), "→", dim2.count())                        # tăng đúng 1

dim_model: 2167 → 2168


In [41]:
spark.stop()
# fresh(SANDBOX / ...)